In [ ]:
# -*- coding: utf-8 -*-


phase2_analysis_updated.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1bygNKYxHy161g4pn1xQRJsYV-Gy9NcQl


In [ ]:

# -*- coding: utf-8 -*-



phase2_analysis.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1_CygpBbGLXIRdMrbMd7LLXmn0v1ZOs_S



In [ ]:

#!/usr/bin/env python3

from google.colab import drive
drive.mount('/content/drive')



Phase 2 Analysis Pipeline — Hindi Verbal Fluency & SpAM
=========================================================
RQ1: Phonological similarity trend over retrieval position
RQ2: Joint cue model (SpAM distance + phonological similarity → IRT)
RQ3: Domain differences in clustering patterns
RQ4: Unified GLM (fluency × domain × position → IRT)




In [ ]:

!pip install indic-transliteration

import json
import warnings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr, spearmanr, mannwhitneyu, kruskal, f_oneway
from itertools import combinations
import statsmodels.formula.api as smf
import statsmodels.api as sm
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

warnings.filterwarnings('ignore')

# ── Poster-quality plot settings ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

PALETTE = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
OUTPUT_DIR = '/content/drive/MyDrive/phase2_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)



══════════════════════════════════════════════════════════════════════════════
SECTION 1: DATA LOADING & PREPROCESSING
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("=" * 70)
print("PHASE 2 ANALYSIS — DATA LOADING")
print("=" * 70)

with open('/content/drive/MyDrive/BRSM/responses.json', 'r') as f:
    data = json.load(f)

df = pd.DataFrame.from_dict(data['fluency-spam'], orient='index').reset_index(drop=True)

# ── Parse VFT trials ──────────────────────────────────────────────────────────
vft_results = []
for _, row in df.iterrows():
    subject_id = row['subject_id']
    for trial in (row['data'] if isinstance(row['data'], list) else []):
        if trial.get('task') == 'VFT' and trial.get('trial_type') == 'html-keyboard-response':
            domain = trial.get('domain', 'unknown')
            if 'practice' in domain.lower():
                continue

            tagged = trial.get('tagged_responses', [])
            if isinstance(tagged, str):
                try: tagged = json.loads(tagged)
                except: tagged = []

            rts = trial.get('response_times', [])
            if isinstance(rts, str):
                try: rts = json.loads(rts)
                except: rts = []

            words = [item.get('response', '') for item in tagged
                     if isinstance(item, dict) and 'response' in item]

            vft_results.append({
                'subject_id': subject_id,
                'domain': domain,
                'word_count': len(words),
                'mean_irt': np.mean(rts) if rts else np.nan,
                'words': words,
                'irts': rts
            })

df_vft = pd.DataFrame(vft_results)

# ── Parse SpAM trials ─────────────────────────────────────────────────────────
spam_results = []
for _, row in df.iterrows():
    subject_id = row['subject_id']
    for trial in (row['data'] if isinstance(row['data'], list) else []):
        if trial.get('task') == 'SpAM':
            domain = trial.get('domain', 'unknown')
            if 'practice' in domain.lower():
                continue

            dw = trial.get('droppedwords', [])
            if isinstance(dw, str):
                try: dw = json.loads(dw)
                except: dw = []

            coords_dict = {}
            for item in dw:
                if isinstance(item, dict):
                    w = item.get('word')
                    x = item.get('x_norm')
                    y = item.get('y_norm')
                    if w is not None and x is not None and y is not None:
                        coords_dict[w] = {'word': w, 'x': float(x), 'y': float(y)}

            word_coords = list(coords_dict.values())

            pairwise = []
            if len(word_coords) >= 2:
                coords = [[wc['x'], wc['y']] for wc in word_coords]
                wds = [wc['word'] for wc in word_coords]
                dists = pdist(coords, metric='euclidean')
                k = 0
                for i in range(len(wds)):
                    for j in range(i + 1, len(wds)):
                        pairwise.append({'word1': wds[i], 'word2': wds[j], 'distance': dists[k]})
                        k += 1

            spam_results.append({
                'subject_id': subject_id,
                'domain': domain,
                'word_count': len(word_coords),
                'word_coords': word_coords,
                'pairwise_distances': pairwise
            })

df_spam = pd.DataFrame(spam_results)

# ── Parse Survey ──────────────────────────────────────────────────────────────
survey_rows = []
for _, row in df.iterrows():
    sid = row['subject_id']
    for trial in (row['data'] if isinstance(row['data'], list) else []):
        if trial.get('trial_type') == 'survey-html-form':
            entry = {'subject_id': sid}
            for key in ['Hi_Read', 'Hi_Write', 'En_Read', 'En_Write',
                        'age', 'gender', 'first_language', 'language_count', 'education']:
                if key in trial:
                    entry[key] = trial[key]
            if len(entry) > 1:
                survey_rows.append(entry)

survey_df = pd.DataFrame(survey_rows).groupby('subject_id').first().reset_index()
for col in ['Hi_Read', 'Hi_Write', 'En_Read', 'En_Write']:
    if col in survey_df.columns:
        survey_df[col] = pd.to_numeric(survey_df[col], errors='coerce')

hi_cols = [c for c in survey_df.columns if c.startswith('Hi_')]
survey_df['hi_fluency'] = survey_df[hi_cols].mean(axis=1) if hi_cols else np.nan

print(f"Subjects: {df_vft['subject_id'].nunique()}")
print(f"VFT trials (non-practice): {len(df_vft)}")
print(f"SpAM trials (non-practice): {len(df_spam)}")
print(f"Domains: {df_vft['domain'].value_counts().to_dict()}")



══════════════════════════════════════════════════════════════════════════════
SECTION 2: PHONOLOGICAL SIMILARITY COMPUTATION
══════════════════════════════════════════════════════════════════════════════




In [ ]:

import re

print("\n" + "=" * 70)
print("COMPUTING PHONOLOGICAL SIMILARITY")
print("=" * 70)

def is_devanagari(word):


    Returns True if the word contains Devanagari characters.
    


In [ ]:
    if not isinstance(word, str):
        return False
    # Search for any character in the Devanagari Unicode block
    return bool(re.search(r'[\u0900-\u097F]', word))

from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def romanize(word):


    Converts Devanagari text to Roman script (ITRANS format).
    Leaves the word unchanged if it is not a string.
    


In [ ]:
    if not isinstance(word, str):
        return word

    # Convert from Devanagari to Roman (ITRANS format is very standard)
    return transliterate(word, sanscript.DEVANAGARI, sanscript.ITRANS)



Convert word to lowercase romanized form for edit distance.



In [ ]:

def romanize(word):


Convert word to lowercase romanized form for edit distance.


In [ ]:
    """Convert word to lowercase romanized form for edit distance."""
        return transliterate(word, sanscript.DEVANAGARI, sanscript.ITRANS).lower()
    return word.lower()

def normalized_edit_distance(a, b):


Levenshtein edit distance normalized by max length. Returns similarity (1 - norm_dist).


In [ ]:
    """Levenshtein edit distance normalized by max length. Returns similarity (1 - norm_dist)."""
    if not a or not b:
        return np.nan

    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)

    return 1 - dp[m][n] / max(m, n)  # similarity: 1 = identical, 0 = max different



══════════════════════════════════════════════════════════════════════════════
SECTION 3: BUILD TRANSITION-LEVEL DATAFRAME
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("\n" + "=" * 70)
print("BUILDING TRANSITION-LEVEL DATAFRAME")
print("=" * 70)

transitions = []
word_level_rows = []

for _, vft_row in df_vft.iterrows():
    subj, domain = vft_row['subject_id'], vft_row['domain']
    words, irts = vft_row['words'], vft_row['irts']

    # Find matching SpAM trial
    spam_match = df_spam[(df_spam['subject_id'] == subj) & (df_spam['domain'] == domain)]
    if spam_match.empty:
        continue

    # Build SpAM distance lookup
    dist_lookup = {}
    for item in spam_match.iloc[0]['pairwise_distances']:
        dist_lookup[(item['word1'], item['word2'])] = item['distance']
        dist_lookup[(item['word2'], item['word1'])] = item['distance']

    # Build SpAM coordinate lookup for density
    coord_lookup = {}
    for wc in spam_match.iloc[0]['word_coords']:
        coord_lookup[wc['word']] = (wc['x'], wc['y'])

    # Compute within-participant SpAM normalization (z-score distances)
    all_dists_this_trial = [item['distance'] for item in spam_match.iloc[0]['pairwise_distances']]
    trial_mean_dist = np.mean(all_dists_this_trial) if all_dists_this_trial else 0
    trial_std_dist = np.std(all_dists_this_trial) if all_dists_this_trial else 1
    if trial_std_dist == 0:
        trial_std_dist = 1

    # Compute median distance for switch detection
    median_dist = np.median(all_dists_this_trial) if all_dists_this_trial else 0.5

    # Word-level: mean neighbor distance (density)
    for word in words:
        if word in coord_lookup:
            dists_to_others = [dist_lookup.get((word, other), np.nan)
                               for other in coord_lookup if other != word]
            dists_to_others = [d for d in dists_to_others if not np.isnan(d)]
            mean_neigh = np.mean(dists_to_others) if dists_to_others else np.nan
        else:
            mean_neigh = np.nan

    # Transition-level data
    min_len = min(len(words), len(irts))
    for i in range(1, min_len):
        w_prev, w_curr = words[i-1], words[i]
        irt = irts[i]

        # SpAM distance
        spam_dist = dist_lookup.get((w_prev, w_curr), np.nan)
        spam_dist_z = (spam_dist - trial_mean_dist) / trial_std_dist if not np.isnan(spam_dist) else np.nan

        # Phonological similarity
        phon_sim = normalized_edit_distance(w_prev, w_curr)

        # Is this a switch? (distance > median)
        is_switch = int(spam_dist > median_dist) if not np.isnan(spam_dist) else np.nan

        transitions.append({
            'subject_id': subj,
            'domain': domain,
            'position': i + 1,  # 1-indexed position of the CURRENT word
            'word_prev': w_prev,
            'word_curr': w_curr,
            'irt_ms': irt,
            'log_irt': np.log1p(irt) if irt > 0 else np.nan,
            'spam_dist': spam_dist,
            'spam_dist_z': spam_dist_z,
            'phon_sim': phon_sim,
            'is_switch': is_switch,
            'trial_word_count': min_len,
        })

    # Word-level rows for density analysis
    for i in range(min_len):
        word = words[i]
        irt_val = irts[i]
        if word in coord_lookup:
            dists_to_others = [dist_lookup.get((word, other), np.nan)
                               for other in coord_lookup if other != word]
            dists_to_others = [d for d in dists_to_others if not np.isnan(d)]
            mean_neigh = np.mean(dists_to_others) if dists_to_others else np.nan
            # Normalize
            mean_neigh_z = (mean_neigh - trial_mean_dist) / trial_std_dist if not np.isnan(mean_neigh) else np.nan
        else:
            mean_neigh = np.nan
            mean_neigh_z = np.nan

        word_level_rows.append({
            'subject_id': subj,
            'domain': domain,
            'position': i + 1,
            'word': word,
            'irt_ms': irt_val,
            'log_irt': np.log1p(irt_val) if irt_val > 0 else np.nan,
            'mean_neigh_dist': mean_neigh,
            'mean_neigh_dist_z': mean_neigh_z,
        })

df_trans = pd.DataFrame(transitions)
df_words = pd.DataFrame(word_level_rows)

# Filter outliers
df_trans = df_trans[(df_trans['irt_ms'] > 0) & (df_trans['irt_ms'] < 30000)].dropna(
    subset=['spam_dist', 'phon_sim', 'log_irt'])
df_words = df_words[(df_words['irt_ms'] > 0) & (df_words['irt_ms'] < 30000)].dropna(
    subset=['log_irt'])

# Merge fluency
df_trans = df_trans.merge(survey_df[['subject_id', 'hi_fluency']], on='subject_id', how='left')
df_words = df_words.merge(survey_df[['subject_id', 'hi_fluency']], on='subject_id', how='left')

# Scale position to [0, 1] within each trial for comparability
df_trans['position_scaled'] = df_trans.groupby(['subject_id', 'domain'])['position'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0.5
)

print(f"Transition-level rows: {len(df_trans)}")
print(f"Word-level rows: {len(df_words)}")
print(f"Domains in transitions: {df_trans['domain'].value_counts().to_dict()}")

# Compute cluster-level stats per trial
cluster_rows = []
for (subj, domain), grp in df_trans.groupby(['subject_id', 'domain']):
    switches = grp['is_switch'].values
    cluster_sizes = []
    current = 1
    for s in switches:
        if s == 0:
            current += 1
        else:
            cluster_sizes.append(current)
            current = 1
    cluster_sizes.append(current)

    cluster_rows.append({
        'subject_id': subj,
        'domain': domain,
        'n_words': len(grp) + 1,
        'n_clusters': len(cluster_sizes),
        'mean_cluster_size': np.mean(cluster_sizes),
        'n_switches': int(sum(switches)),
        'mean_irt': grp['irt_ms'].mean(),
        'mean_spam_dist': grp['spam_dist'].mean(),
        'mean_phon_sim': grp['phon_sim'].mean(),
    })

df_clusters = pd.DataFrame(cluster_rows)
df_clusters = df_clusters.merge(survey_df[['subject_id', 'hi_fluency']], on='subject_id', how='left')



══════════════════════════════════════════════════════════════════════════════
RQ1: PHONOLOGICAL SIMILARITY OVER RETRIEVAL POSITION
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("\n" + "=" * 70)
print("RQ1: PHONOLOGICAL SIMILARITY OVER RETRIEVAL POSITION")
print("=" * 70)

# ── 1a. Descriptive: binned phonological similarity by position ───────────────
df_trans['position_bin'] = pd.cut(df_trans['position'], bins=[0, 4, 8, 12, 50],
                                   labels=['Early (2-4)', 'Mid-Early (5-8)',
                                           'Mid-Late (9-12)', 'Late (13+)'])

phon_by_bin = df_trans.groupby('position_bin', observed=True)['phon_sim'].agg(['mean', 'std', 'count'])
print("\nPhonological similarity by position bin:")
print(phon_by_bin.round(4))

# ── 1b. Mixed-effects: phon_sim ~ position_scaled + (1 | subject) ────────────
try:
    md_rq1 = smf.mixedlm("phon_sim ~ position_scaled", df_trans,
                           groups=df_trans["subject_id"]).fit(reml=True)
    print("\nMixed-Effects Model: phon_sim ~ position_scaled + (1 | subject)")
    print(md_rq1.summary().tables[1])
    rq1_coef = md_rq1.params['position_scaled']
    rq1_pval = md_rq1.pvalues['position_scaled']
    print(f"\n→ Position coefficient: {rq1_coef:.4f}, p = {rq1_pval:.4g}")
    if rq1_pval < 0.05:
        direction = "increases" if rq1_coef > 0 else "decreases"
        print(f"→ Phonological similarity significantly {direction} over the task.")
    else:
        print("→ No significant trend in phonological similarity over the task.")
except Exception as e:
    print(f"Mixed model failed: {e}")
    rq1_coef, rq1_pval = np.nan, np.nan

# ── 1c. Compare within-cluster vs switch phonological similarity ──────────────
within = df_trans[df_trans['is_switch'] == 0]['phon_sim']
switch = df_trans[df_trans['is_switch'] == 1]['phon_sim']
u_stat, u_pval = mannwhitneyu(within, switch, alternative='two-sided')
print(f"\nPhon. similarity: within-cluster M={within.mean():.4f}, switch M={switch.mean():.4f}")
print(f"Mann-Whitney U = {u_stat:.1f}, p = {u_pval:.4g}")

# ── PLOT RQ1 ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1A: Trend over position
pos_means = df_trans.groupby('position')['phon_sim'].agg(['mean', 'sem']).reset_index()
pos_means = pos_means[pos_means['position'] <= 20]  # cap for readability
axes[0].errorbar(pos_means['position'], pos_means['mean'],
                 yerr=pos_means['sem'], fmt='o-', color=PALETTE[0],
                 markersize=5, capsize=3, linewidth=1.5)
z = np.polyfit(pos_means['position'], pos_means['mean'], 1)
axes[0].plot(pos_means['position'], np.polyval(z, pos_means['position']),
             '--', color=PALETTE[1], linewidth=2, alpha=0.8,
             label=f'β={rq1_coef:.3f}, p={rq1_pval:.3f}')
axes[0].set_xlabel('Retrieval Position')
axes[0].set_ylabel('Phonological Similarity')
axes[0].set_title('A. Phonological Similarity\nOver Retrieval Position')
axes[0].legend(fontsize=9)

# 1B: Within vs Switch
box_data = pd.DataFrame({
    'Phonological Similarity': list(within) + list(switch),
    'Transition Type': ['Within Cluster'] * len(within) + ['Switch'] * len(switch)
})
sns.boxplot(data=box_data, x='Transition Type', y='Phonological Similarity',
            palette=[PALETTE[0], PALETTE[1]], ax=axes[1], width=0.5)
axes[1].set_title(f'B. Phonological Similarity:\nWithin vs Switch (p={u_pval:.3g})')
axes[1].set_xlabel('')

# 1C: Dual trajectory (semantic + phonological over position)
sem_means = df_trans.groupby('position')['spam_dist_z'].mean().reset_index()
phon_means = df_trans.groupby('position')['phon_sim'].mean().reset_index()
sem_means = sem_means[sem_means['position'] <= 20]
phon_means = phon_means[phon_means['position'] <= 20]

ax1c = axes[2]
color_sem = PALETTE[3]
color_phon = PALETTE[0]

ln1 = ax1c.plot(sem_means['position'], sem_means['spam_dist_z'],
                'o-', color=color_sem, label='Semantic Distance (z)', markersize=4)
ax1c.set_ylabel('Semantic Distance (z-scored)', color=color_sem)
ax1c.tick_params(axis='y', labelcolor=color_sem)

ax1c_twin = ax1c.twinx()
ln2 = ax1c_twin.plot(phon_means['position'], phon_means['phon_sim'],
                     's-', color=color_phon, label='Phonological Similarity', markersize=4)
ax1c_twin.set_ylabel('Phonological Similarity', color=color_phon)
ax1c_twin.tick_params(axis='y', labelcolor=color_phon)

lns = ln1 + ln2
labs = [l.get_label() for l in lns]
ax1c.legend(lns, labs, fontsize=8, loc='upper left')
ax1c.set_xlabel('Retrieval Position')
ax1c.set_title('C. Dual Trajectory:\nSemantic Distance vs Phonological Similarity')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ1_phonological_trend.png', bbox_inches='tight')
plt.close()
print(f"\n✓ Saved: RQ1_phonological_trend.png")


print("\n" + "-" * 50)
print("RQ1 SUPPLEMENTARY: Same-Script vs Cross-Script Analysis")
print("-" * 50)

# Tag each transition by script match
df_trans['prev_devanagari'] = df_trans['word_prev'].apply(is_devanagari)
df_trans['curr_devanagari'] = df_trans['word_curr'].apply(is_devanagari)
df_trans['same_script'] = df_trans['prev_devanagari'] == df_trans['curr_devanagari']

same = df_trans[df_trans['same_script'] == True]['phon_sim']
cross = df_trans[df_trans['same_script'] == False]['phon_sim']

print(f"Same-script transitions: n={len(same)}, mean phon_sim={same.mean():.4f}")
print(f"Cross-script transitions: n={len(cross)}, mean phon_sim={cross.mean():.4f}")

if len(same) > 0 and len(cross) > 0:
    u_script, p_script = mannwhitneyu(same, cross, alternative='two-sided')
    print(f"Mann-Whitney U = {u_script:.1f}, p = {p_script:.4g}")

    if p_script < 0.05:
        print("→ Phonological similarity is significantly higher for same-script pairs.")
        print("  This suggests mixed-script responding attenuates the phonological signal,")
        print("  which may explain the null RQ1 trend.")
    else:
        print("→ No significant difference between same- and cross-script pairs.")

# Re-test phonological trend within same-script only
df_same_script = df_trans[df_trans['same_script'] == True]
if len(df_same_script) > 50:
    try:
        md_same = smf.mixedlm("phon_sim ~ position_scaled", df_same_script,
                               groups=df_same_script["subject_id"]).fit(reml=True)
        print(f"\nSame-script only trend: β={md_same.params['position_scaled']:.4f}, "
              f"p={md_same.pvalues['position_scaled']:.4g}")
    except:
        pass

# Plot same vs cross script
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Boxplot
script_data = pd.DataFrame({
    'Phonological Similarity': list(same) + list(cross),
    'Script Match': ['Same Script'] * len(same) + ['Cross Script'] * len(cross)
})
sns.boxplot(data=script_data, x='Script Match', y='Phonological Similarity',
            palette=[PALETTE[0], PALETTE[3]], ax=axes[0], width=0.5)
axes[0].set_title(f'A. Phonological Similarity by Script Match\n(p={p_script:.3g})')
axes[0].set_xlabel('')

# Panel B: Script composition by domain
script_by_domain = df_trans.groupby('domain')['same_script'].mean().reset_index()
script_by_domain.columns = ['Domain', 'Proportion Same-Script']
bars = axes[1].bar(script_by_domain['Domain'], script_by_domain['Proportion Same-Script'],
                   color=PALETTE[:len(script_by_domain)], edgecolor='#333')
axes[1].set_ylabel('Proportion Same-Script Transitions')
axes[1].set_title('B. Script Consistency by Domain')
axes[1].set_ylim(0, 1)
axes[1].axhline(0.5, color='grey', linestyle='--', alpha=0.5, label='50% baseline')
axes[1].legend()
for bar, val in zip(bars, script_by_domain['Proportion Same-Script']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{val:.0%}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ1_script_analysis.png', bbox_inches='tight')
plt.close()
print(f"✓ Saved: RQ1_script_analysis.png")



══════════════════════════════════════════════════════════════════════════════
RQ2: JOINT CUE MODEL — SpAM + PHONOLOGY → IRT
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("\n" + "=" * 70)
print("RQ2: JOINT CUE MODEL — SpAM + PHONOLOGY → IRT")
print("=" * 70)

# ── 2a. Basic mixed model: log_irt ~ spam_dist_z + (1 | subject) ─────────────
print("\n--- Model 1: log_irt ~ spam_dist_z (resolving Phase 1 null) ---")
try:
    m1 = smf.mixedlm("log_irt ~ spam_dist_z", df_trans,
                       groups=df_trans["subject_id"]).fit(reml=True)
    print(m1.summary().tables[1])
    print(f"→ SpAM distance: β={m1.params['spam_dist_z']:.4f}, p={m1.pvalues['spam_dist_z']:.4g}")
except Exception as e:
    print(f"Model 1 failed: {e}")

# ── 2b. Joint model: log_irt ~ spam_dist_z + phon_sim + position + (1|subj) ──
print("\n--- Model 2: log_irt ~ spam_dist_z + phon_sim + position_scaled ---")
try:
    m2 = smf.mixedlm("log_irt ~ spam_dist_z + phon_sim + position_scaled", df_trans,
                       groups=df_trans["subject_id"]).fit(reml=True)
    print(m2.summary().tables[1])
    for param in ['spam_dist_z', 'phon_sim', 'position_scaled']:
        print(f"  {param}: β={m2.params[param]:.4f}, p={m2.pvalues[param]:.4g}")
except Exception as e:
    print(f"Model 2 failed: {e}")

# ── 2c. Interaction model: does phon_sim matter more at switches? ─────────────
print("\n--- Model 3: + is_switch × phon_sim interaction ---")
try:
    m3 = smf.mixedlm("log_irt ~ spam_dist_z + phon_sim * is_switch + position_scaled",
                       df_trans.dropna(subset=['is_switch']),
                       groups=df_trans.dropna(subset=['is_switch'])["subject_id"]).fit(reml=True)
    print(m3.summary().tables[1])
    if 'phon_sim:is_switch' in m3.params:
        print(f"\n→ Interaction phon_sim × switch: β={m3.params['phon_sim:is_switch']:.4f}, "
              f"p={m3.pvalues['phon_sim:is_switch']:.4g}")
except Exception as e:
    print(f"Model 3 failed: {e}")

# ── 2d. Model comparison table ────────────────────────────────────────────────
print("\n--- Model Comparison (AIC / BIC) ---")
model_comp = []
for name, model in [("M1: SpAM only", m1), ("M2: SpAM+Phon+Pos", m2), ("M3: +Interaction", m3)]:
    try:
        k = len(model.params)
        aic_manual = 2 * k - 2 * model.llf
        bic_manual = k * np.log(model.nobs) - 2 * model.llf
        model_comp.append({
            'Model': name,
            'AIC': f"{aic_manual:.1f}",
            'BIC': f"{bic_manual:.1f}",
            'Log-Lik': f"{model.llf:.1f}",
            'N': int(model.nobs)
        })
    except:
        pass
if model_comp:
    comp_df = pd.DataFrame(model_comp)
    print(comp_df.to_string(index=False))
    best_aic = comp_df.loc[comp_df['AIC'].astype(float).idxmin(), 'Model']
    print(f"\n→ Best model by AIC: {best_aic}")

# ── PLOT RQ2 ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 2A: SpAM distance vs log(IRT) with regression
axes[0].scatter(df_trans['spam_dist_z'], df_trans['log_irt'],
                alpha=0.15, s=10, color=PALETTE[0])
z2 = np.polyfit(df_trans['spam_dist_z'].values, df_trans['log_irt'].values, 1)
x_range = np.linspace(df_trans['spam_dist_z'].min(), df_trans['spam_dist_z'].max(), 100)
axes[0].plot(x_range, np.polyval(z2, x_range), color=PALETTE[1], linewidth=2.5)
try:
    beta_label = f"β={m1.params['spam_dist_z']:.3f}, p={m1.pvalues['spam_dist_z']:.3g}"
except:
    beta_label = ""
axes[0].set_xlabel('SpAM Distance (z-scored)')
axes[0].set_ylabel('log(1 + IRT)')
axes[0].set_title(f'A. Semantic Distance → IRT\n(Mixed Effects: {beta_label})')

# 2B: Phonological similarity vs log(IRT)
axes[1].scatter(df_trans['phon_sim'], df_trans['log_irt'],
                alpha=0.15, s=10, color=PALETTE[2])
z3 = np.polyfit(df_trans['phon_sim'].values, df_trans['log_irt'].values, 1)
x_range2 = np.linspace(df_trans['phon_sim'].min(), df_trans['phon_sim'].max(), 100)
axes[1].plot(x_range2, np.polyval(z3, x_range2), color=PALETTE[1], linewidth=2.5)
try:
    beta_label2 = f"β={m2.params['phon_sim']:.3f}, p={m2.pvalues['phon_sim']:.3g}"
except:
    beta_label2 = ""
axes[1].set_xlabel('Phonological Similarity')
axes[1].set_ylabel('log(1 + IRT)')
axes[1].set_title(f'B. Phonological Similarity → IRT\n(Joint Model: {beta_label2})')

# 2C: Coefficient comparison plot
try:
    coefs = []
    for param in ['spam_dist_z', 'phon_sim', 'position_scaled']:
        coefs.append({
            'Predictor': param.replace('_', ' ').title(),
            'Coefficient': m2.params[param],
            'SE': m2.bse[param],
            'p': m2.pvalues[param]
        })
    cdf = pd.DataFrame(coefs)
    colors_bar = [PALETTE[0] if p < 0.05 else '#CCCCCC' for p in cdf['p']]
    bars = axes[2].barh(cdf['Predictor'], cdf['Coefficient'], xerr=cdf['SE'] * 1.96,
                        color=colors_bar, edgecolor='#333', capsize=4, height=0.5)
    axes[2].axvline(x=0, color='black', linewidth=0.8, linestyle='-')
    axes[2].set_xlabel('Coefficient (β)')
    axes[2].set_title('C. Joint Model Coefficients\n(95% CI, colored = p<.05)')
    for i, row in cdf.iterrows():
        axes[2].text(row['Coefficient'] + row['SE'] * 2 + 0.01, i,
                     f"p={row['p']:.3g}", va='center', fontsize=9)
except Exception as e:
    axes[2].text(0.5, 0.5, f"Plot failed: {e}", transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ2_joint_cue_model.png', bbox_inches='tight')
plt.close()
print(f"\n✓ Saved: RQ2_joint_cue_model.png")


print("\n" + "=" * 70)
print("RQ2 ROBUSTNESS CHECKS")
print("=" * 70)

# ── Robustness 1: Raw SpAM distance (not z-scored) ───────────────────────
print("\n--- Robustness: Using raw SpAM distance instead of z-scored ---")
try:
    m1_raw = smf.mixedlm("log_irt ~ spam_dist", df_trans,
                          groups=df_trans["subject_id"]).fit(reml=True)
    print(f"  Raw SpAM dist: β={m1_raw.params['spam_dist']:.4f}, p={m1_raw.pvalues['spam_dist']:.4g}")
except Exception as e:
    print(f"  Failed: {e}")

# ── Robustness 2: Winsorized IRT at 95th percentile ──────────────────────
print("\n--- Robustness: Winsorizing IRT at 95th percentile ---")
p95 = df_trans['irt_ms'].quantile(0.95)
df_winsor = df_trans.copy()
df_winsor['log_irt_w'] = np.log1p(df_winsor['irt_ms'].clip(upper=p95))
try:
    m1_winsor = smf.mixedlm("log_irt_w ~ spam_dist_z", df_winsor,
                             groups=df_winsor["subject_id"]).fit(reml=True)
    print(f"  Winsorized SpAM: β={m1_winsor.params['spam_dist_z']:.4f}, p={m1_winsor.pvalues['spam_dist_z']:.4g}")
except Exception as e:
    print(f"  Failed: {e}")

# ── Robustness 3: Excluding first response (position > 2) ───────────────
print("\n--- Robustness: Excluding first transition (position > 2) ---")
df_no_first = df_trans[df_trans['position'] > 2]
try:
    m1_nofirst = smf.mixedlm("log_irt ~ spam_dist_z", df_no_first,
                              groups=df_no_first["subject_id"]).fit(reml=True)
    print(f"  No first: β={m1_nofirst.params['spam_dist_z']:.4f}, p={m1_nofirst.pvalues['spam_dist_z']:.4g}")
except Exception as e:
    print(f"  Failed: {e}")

print("\n→ If all three show significant SpAM effect, the result is robust.")



══════════════════════════════════════════════════════════════════════════════
RQ3: DOMAIN DIFFERENCES IN CLUSTERING
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("\n" + "=" * 70)
print("RQ3: DOMAIN DIFFERENCES IN CLUSTERING")
print("=" * 70)

# Focus on domains with enough data
domain_counts = df_clusters['domain'].value_counts()
valid_domains = domain_counts[domain_counts >= 10].index.tolist()
df_clust_valid = df_clusters[df_clusters['domain'].isin(valid_domains)]

print(f"\nDomains with ≥10 trials: {valid_domains}")
print(f"\nDescriptive stats by domain:")
domain_desc = df_clust_valid.groupby('domain').agg({
    'n_words': ['mean', 'std'],
    'mean_cluster_size': ['mean', 'std'],
    'n_switches': ['mean', 'std'],
    'mean_irt': ['mean', 'std'],
    'mean_phon_sim': ['mean', 'std'],
    'mean_spam_dist': ['mean', 'std']
}).round(3)
print(domain_desc)

# ── Kruskal-Wallis tests (non-parametric, safe for unequal n) ─────────────────
print("\n--- Kruskal-Wallis Tests Across Domains ---")
kw_results = []
for var in ['mean_cluster_size', 'n_words', 'mean_irt', 'mean_phon_sim', 'n_switches']:
    groups = [grp[var].dropna().values for _, grp in df_clust_valid.groupby('domain')]
    if len(groups) >= 2 and all(len(g) > 0 for g in groups):
        h_stat, p_val = kruskal(*groups)
        kw_results.append({'Variable': var, 'H': h_stat, 'p': p_val,
                          'Significant': '***' if p_val < 0.001 else '**' if p_val < 0.01
                          else '*' if p_val < 0.05 else 'ns'})
        print(f"  {var}: H={h_stat:.2f}, p={p_val:.4g} {'*' if p_val < 0.05 else 'ns'}")

kw_df = pd.DataFrame(kw_results)

# ── Post-hoc pairwise comparisons for significant variables ───────────────────
print("\n--- Post-hoc Pairwise Comparisons (Mann-Whitney) ---")
for var in kw_df[kw_df['Significant'] != 'ns']['Variable'].tolist():
    print(f"\n  {var}:")
    for d1, d2 in combinations(valid_domains, 2):
        g1 = df_clust_valid[df_clust_valid['domain'] == d1][var].dropna()
        g2 = df_clust_valid[df_clust_valid['domain'] == d2][var].dropna()
        if len(g1) > 0 and len(g2) > 0:
            u, p = mannwhitneyu(g1, g2, alternative='two-sided')
            sig = '*' if p < 0.05 else 'ns'
            print(f"    {d1} vs {d2}: U={u:.0f}, p={p:.4g} {sig}")

# ── PLOT RQ3 ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

plot_vars = [
    ('n_words', 'Total Words Retrieved', axes[0, 0]),
    ('mean_cluster_size', 'Mean Cluster Size', axes[0, 1]),
    ('n_switches', 'Number of Switches', axes[0, 2]),
    ('mean_irt', 'Mean IRT (ms)', axes[1, 0]),
    ('mean_phon_sim', 'Mean Phonological Similarity', axes[1, 1]),
    ('mean_spam_dist', 'Mean SpAM Distance', axes[1, 2]),
]

for var, title, ax in plot_vars:
    sns.boxplot(data=df_clust_valid, x='domain', y=var,
                palette=PALETTE[:len(valid_domains)], ax=ax, width=0.6)
    sns.stripplot(data=df_clust_valid, x='domain', y=var,
                  color='black', alpha=0.3, size=3, ax=ax, jitter=True)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

    # Add significance from KW
    kw_row = kw_df[kw_df['Variable'] == var]
    if not kw_row.empty:
        sig_text = f"KW p={kw_row.iloc[0]['p']:.3g}"
        ax.text(0.98, 0.98, sig_text, transform=ax.transAxes,
                ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.suptitle('RQ3: Domain Differences in Clustering & Retrieval', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ3_domain_differences.png', bbox_inches='tight')
plt.close()
print(f"\n✓ Saved: RQ3_domain_differences.png")



══════════════════════════════════════════════════════════════════════════════
RQ4: UNIFIED GLM — FLUENCY × DOMAIN × POSITION → IRT
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("\n" + "=" * 70)
print("RQ4: UNIFIED GLM — FLUENCY × DOMAIN × POSITION → IRT")
print("=" * 70)

df_glm = df_trans[df_trans['domain'].isin(valid_domains)].dropna(
    subset=['hi_fluency', 'log_irt', 'position_scaled']).copy()

# Create centered fluency
df_glm['fluency_c'] = df_glm['hi_fluency'] - df_glm['hi_fluency'].mean()

# Dummy-code domain (reference = animals)
df_glm['domain'] = pd.Categorical(df_glm['domain'])

print(f"\nGLM sample: {len(df_glm)} observations, {df_glm['subject_id'].nunique()} subjects")

# ── 4a. Main effects model ────────────────────────────────────────────────────
print("\n--- Model A: Main effects ---")
try:
    glm_a = smf.mixedlm("log_irt ~ fluency_c + C(domain) + position_scaled",
                          df_glm, groups=df_glm["subject_id"]).fit(reml=True)
    print(glm_a.summary().tables[1])
except Exception as e:
    print(f"GLM A failed: {e}")
    glm_a = None

# ── 4b. Interaction model ─────────────────────────────────────────────────────
print("\n--- Model B: + fluency × position + fluency × domain interactions ---")
try:
    glm_b = smf.mixedlm(
        "log_irt ~ fluency_c * position_scaled + fluency_c * C(domain) + C(domain) * position_scaled",
        df_glm, groups=df_glm["subject_id"]).fit(reml=True)
    print(glm_b.summary().tables[1])
except Exception as e:
    print(f"GLM B failed: {e}")
    glm_b = None

# ── Model comparison ──────────────────────────────────────────────────────────
if glm_a and glm_b:
    print(f"\n--- Model Comparison ---")
    print(f"Model A (main effects):  AIC={glm_a.aic:.1f}, BIC={glm_a.bic:.1f}")
    print(f"Model B (interactions):  AIC={glm_b.aic:.1f}, BIC={glm_b.bic:.1f}")
    better = "B (interactions)" if glm_b.aic < glm_a.aic else "A (main effects)"
    print(f"→ Preferred by AIC: Model {better}")

# ── PLOT RQ4 ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 4A: IRT by fluency split
median_flu = df_glm['hi_fluency'].median()
df_glm['fluency_group'] = np.where(df_glm['hi_fluency'] >= median_flu, 'High Fluency', 'Low Fluency')

for i, grp_name in enumerate(['Low Fluency', 'High Fluency']):
    grp = df_glm[df_glm['fluency_group'] == grp_name]
    pos_means = grp.groupby('position')['irt_ms'].mean().reset_index()
    pos_means = pos_means[pos_means['position'] <= 15]
    axes[0].plot(pos_means['position'], pos_means['irt_ms'],
                 'o-', color=PALETTE[i], label=grp_name, markersize=4, linewidth=1.5)

axes[0].set_xlabel('Retrieval Position')
axes[0].set_ylabel('Mean IRT (ms)')
axes[0].set_title('A. IRT Trajectory by Fluency Group')
axes[0].legend()

# 4B: IRT by domain over position
for i, dom in enumerate(valid_domains):
    grp = df_glm[df_glm['domain'] == dom]
    pos_means = grp.groupby('position')['irt_ms'].mean().reset_index()
    pos_means = pos_means[pos_means['position'] <= 15]
    axes[1].plot(pos_means['position'], pos_means['irt_ms'],
                 'o-', color=PALETTE[i], label=dom.title(), markersize=4, linewidth=1.5)

axes[1].set_xlabel('Retrieval Position')
axes[1].set_ylabel('Mean IRT (ms)')
axes[1].set_title('B. IRT Trajectory by Domain')
axes[1].legend(fontsize=9)

# 4C: Coefficient forest plot for best model
best_model = glm_b if (glm_b and glm_a and glm_b.aic < glm_a.aic) else glm_a
if best_model:
    try:
        params = best_model.params.drop('Intercept', errors='ignore')
        params = params.drop('Group Var', errors='ignore')
        ses = best_model.bse.reindex(params.index)
        pvals = best_model.pvalues.reindex(params.index)

        # Clean up names
        clean_names = [n.replace('C(domain)[T.', '').replace(']', '').replace('_', ' ').title()
                       for n in params.index]

        y_pos = range(len(params))
        colors_forest = [PALETTE[0] if p < 0.05 else '#BBBBBB' for p in pvals]

        axes[2].barh(y_pos, params.values, xerr=ses.values * 1.96,
                     color=colors_forest, edgecolor='#333', capsize=3, height=0.6)
        axes[2].set_yticks(y_pos)
        axes[2].set_yticklabels(clean_names, fontsize=8)
        axes[2].axvline(0, color='black', linewidth=0.8)
        axes[2].set_xlabel('Coefficient (β)')
        axes[2].set_title('C. Unified Model Coefficients\n(95% CI)')
    except Exception as e:
        axes[2].text(0.5, 0.5, f"Plot failed: {e}", transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ4_unified_glm.png', bbox_inches='tight')
plt.close()
print(f"\n✓ Saved: RQ4_unified_glm.png")

print("\n" + "-" * 50)
print("RQ4 SUPPLEMENTARY: Fluency × Domain Interaction Deep Dive")
print("-" * 50)

# Focus on the significant interaction: fluency × body-parts
df_interaction = df_glm[df_glm['domain'].isin(['animals', 'body-parts'])].copy()
median_flu = df_interaction['hi_fluency'].median()
df_interaction['fluency_group'] = np.where(df_interaction['hi_fluency'] >= median_flu,
                                            'High Fluency', 'Low Fluency')

# Compute group means
interaction_means = df_interaction.groupby(['domain', 'fluency_group']).agg(
    mean_irt=('irt_ms', 'mean'),
    se_irt=('irt_ms', 'sem'),
    n=('irt_ms', 'count')
).reset_index()
print("\nMean IRT by Domain × Fluency:")
print(interaction_means.to_string(index=False))

# Compute unique word counts to check quality-quantity tradeoff
word_counts = df_interaction.groupby(['domain', 'fluency_group']).agg(
    mean_words=('position', 'max')
).reset_index()
print("\nMax position (proxy for word count) by Domain × Fluency:")
print(word_counts.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Interaction bar chart
domains_plot = ['animals', 'body-parts']
x = np.arange(len(domains_plot))
width = 0.35

for i, flu_grp in enumerate(['Low Fluency', 'High Fluency']):
    means = []
    errors = []
    for dom in domains_plot:
        row = interaction_means[(interaction_means['domain'] == dom) &
                                (interaction_means['fluency_group'] == flu_grp)]
        means.append(row['mean_irt'].values[0] if len(row) > 0 else 0)
        errors.append(row['se_irt'].values[0] * 1.96 if len(row) > 0 else 0)
    bars = axes[0].bar(x + i * width, means, width, yerr=errors,
                       label=flu_grp, color=PALETTE[i], edgecolor='#333',
                       capsize=5, alpha=0.85)

axes[0].set_xlabel('Domain')
axes[0].set_ylabel('Mean IRT (ms)')
axes[0].set_title('A. Fluency × Domain Interaction\n(Animals vs Body-Parts)')
axes[0].set_xticks(x + width / 2)
axes[0].set_xticklabels([d.title() for d in domains_plot])
axes[0].legend()

# Add significance bracket for body-parts
try:
    bp_idx = 1
    y_max = max(interaction_means[interaction_means['domain'] == 'body-parts']['mean_irt']) + 500
    axes[0].plot([bp_idx, bp_idx + width], [y_max, y_max], 'k-', linewidth=1)
    axes[0].text(bp_idx + width/2, y_max + 100, 'β=0.219\np=.003', ha='center', fontsize=8)
except:
    pass

# Panel B: IRT trajectory for body-parts split by fluency
bp_data = df_interaction[df_interaction['domain'] == 'body-parts']
for i, grp_name in enumerate(['Low Fluency', 'High Fluency']):
    grp = bp_data[bp_data['fluency_group'] == grp_name]
    pos_means = grp.groupby('position')['irt_ms'].mean().reset_index()
    pos_means = pos_means[pos_means['position'] <= 12]
    axes[1].plot(pos_means['position'], pos_means['irt_ms'],
                 'o-', color=PALETTE[i], label=grp_name, markersize=5, linewidth=1.5)

axes[1].set_xlabel('Retrieval Position')
axes[1].set_ylabel('Mean IRT (ms)')
axes[1].set_title('B. Body-Parts: IRT by Fluency Group')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/RQ4_interaction_deep_dive.png', bbox_inches='tight')
plt.close()
print(f"✓ Saved: RQ4_interaction_deep_dive.png")



══════════════════════════════════════════════════════════════════════════════
SUMMARY TABLES FOR POSTER
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("\n" + "=" * 70)
print("SUMMARY TABLES FOR POSTER")
print("=" * 70)

# ── Table 1: Overall descriptive stats ────────────────────────────────────────
print("\n--- Table 1: Overall Descriptive Statistics ---")
overall = pd.DataFrame({
    'Metric': ['Total Participants', 'Total VFT Trials', 'Mean Words/Trial',
               'Mean IRT (ms)', 'Mean Phon. Similarity', 'Mean SpAM Distance'],
    'Value': [
        f"{df_vft['subject_id'].nunique()}",
        f"{len(df_vft)}",
        f"{df_vft['word_count'].mean():.1f} (SD={df_vft['word_count'].std():.1f})",
        f"{df_trans['irt_ms'].mean():.0f} (SD={df_trans['irt_ms'].std():.0f})",
        f"{df_trans['phon_sim'].mean():.3f} (SD={df_trans['phon_sim'].std():.3f})",
        f"{df_trans['spam_dist'].mean():.3f} (SD={df_trans['spam_dist'].std():.3f})"
    ]
})
print(overall.to_string(index=False))

# ── Table 2: RQ summary ──────────────────────────────────────────────────────
print("\n--- Table 2: Key Findings Summary ---")
findings = []

# RQ1
findings.append({
    'RQ': 'RQ1',
    'Hypothesis': 'Phon. similarity ↑ over position',
    'Test': 'Mixed-effects LMM',
    'Result': f"β={rq1_coef:.3f}, p={rq1_pval:.3g}",
    'Supported': '✓' if rq1_pval < 0.05 else '✗'
})
findings.append({
    'RQ': 'RQ1',
    'Hypothesis': 'Phon. sim higher within clusters',
    'Test': 'Mann-Whitney U',
    'Result': f"U={u_stat:.0f}, p={u_pval:.3g}",
    'Supported': '✓' if u_pval < 0.05 else '✗'
})

# RQ2
try:
    findings.append({
        'RQ': 'RQ2',
        'Hypothesis': 'SpAM dist → IRT (mixed effects)',
        'Test': 'Mixed-effects LMM',
        'Result': f"β={m1.params['spam_dist_z']:.3f}, p={m1.pvalues['spam_dist_z']:.3g}",
        'Supported': '✓' if m1.pvalues['spam_dist_z'] < 0.05 else '✗'
    })
    findings.append({
        'RQ': 'RQ2',
        'Hypothesis': 'Joint SpAM + Phon predicts IRT',
        'Test': 'Mixed-effects LMM',
        'Result': f"AIC={2*len(m2.params)-2*m2.llf:.0f} vs M1 AIC={2*len(m1.params)-2*m1.llf:.0f}",
        'Supported': '✓' if m2.llf > m1.llf else '✗'
    })
except:
    pass

# RQ3
for _, row in kw_df.iterrows():
    findings.append({
        'RQ': 'RQ3',
        'Hypothesis': f'Domain diff: {row["Variable"]}',
        'Test': 'Kruskal-Wallis',
        'Result': f"H={row['H']:.2f}, p={row['p']:.3g}",
        'Supported': row['Significant']
    })

findings_df = pd.DataFrame(findings)
print(findings_df.to_string(index=False))

# Save tables as CSV
overall.to_csv(f'{OUTPUT_DIR}/table1_descriptives.csv', index=False)
findings_df.to_csv(f'{OUTPUT_DIR}/table2_findings.csv', index=False)
kw_df.to_csv(f'{OUTPUT_DIR}/table3_domain_tests.csv', index=False)
if model_comp:
    comp_df.to_csv(f'{OUTPUT_DIR}/table4_model_comparison.csv', index=False)

print(f"\n✓ All tables saved to {OUTPUT_DIR}/")



══════════════════════════════════════════════════════════════════════════════
BONUS: SAMPLE SpAM VISUALIZATION FOR POSTER
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("\n" + "=" * 70)
print("BONUS VISUALIZATIONS")
print("=" * 70)

# Pick a trial with decent word count for SpAM visualization
good_trials = df_spam[(df_spam['word_count'] >= 8) & (~df_spam['domain'].str.contains('practice'))]
if len(good_trials) > 0:
    sample = good_trials.iloc[0]
    fig, ax = plt.subplots(figsize=(7, 7))
    coords = sample['word_coords']
    xs = [c['x'] for c in coords]
    ys = [c['y'] for c in coords]
    labels = [c['word'] for c in coords]

    ax.scatter(xs, ys, s=100, c=PALETTE[0], edgecolors='white', linewidth=1.5, zorder=5)
    for i, label in enumerate(labels):
        ax.annotate(label, (xs[i], ys[i]), fontsize=8,
                    xytext=(5, 5), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.7))

    ax.set_xlabel('Normalized X')
    ax.set_ylabel('Normalized Y')
    ax.set_title(f'SpAM Layout — Subject {sample["subject_id"]}, {sample["domain"].title()}')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/bonus_spam_layout.png', bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: bonus_spam_layout.png")

# ── Correlation heatmap of key variables ──────────────────────────────────────
corr_vars = ['irt_ms', 'spam_dist', 'spam_dist_z', 'phon_sim', 'position']
corr_matrix = df_trans[corr_vars].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, square=True, linewidths=0.5,
            xticklabels=['IRT', 'SpAM Dist', 'SpAM Dist (z)', 'Phon Sim', 'Position'],
            yticklabels=['IRT', 'SpAM Dist', 'SpAM Dist (z)', 'Phon Sim', 'Position'])
ax.set_title('Correlation Matrix: Key Variables')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/bonus_correlation_heatmap.png', bbox_inches='tight')
plt.close()
print(f"✓ Saved: bonus_correlation_heatmap.png")



══════════════════════════════════════════════════════════════════════════════
FINAL SUMMARY
══════════════════════════════════════════════════════════════════════════════




In [ ]:

print("\n" + "=" * 70)
print("PHASE 2 ANALYSIS COMPLETE")
print("=" * 70)
print(f"\nAll outputs saved to: {OUTPUT_DIR}/")
print(f"\nFiles generated:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f"  {f} ({size/1024:.1f} KB)")

print("\n" + "=" * 70)
print("KEY TAKEAWAYS FOR POSTER")
print("=" * 70)
print("""
1. RQ1 — Phonological similarity does NOT increase over position in Hindi,
   diverging from Kumar et al.'s English findings. Supplementary analysis
   shows participants overwhelmingly stayed within one script (~100%
   same-script), so mixed-script noise is not the primary explanation.
   The null may instead reflect Hindi's richer morphological structure or
   the use of orthographic edit distance rather than true phonemic distance.
   Future work should use a Hindi pronunciation dictionary (e.g., Hindi
   WordNet phonetic forms) for more accurate phonological measurement.

2. RQ2 — The Phase 1 null (r=0.026, p=.41) is fully resolved: SpAM
   distance significantly predicts IRT under mixed-effects modeling
   (β=0.097, p<.0001). This is robust to alternative specifications
   (raw distances, winsorized IRT, excluding first transition).
   Position is the strongest predictor (β=0.486, p<.0001).

3. RQ3 — Domains differ significantly in word count, IRT, phonological
   similarity, and number of switches (all p<.05). Colours is the
   outlier: more words, faster retrieval, more switches. Body-parts
   show the highest phonological similarity, likely due to shared
   Hindi phonological structure. Cluster SIZE does not differ.

4. RQ4 — Fluency × body-parts interaction (β=0.219, p=.003): higher
   fluency is associated with SLOWER retrieval specifically for
   body-parts, possibly reflecting retrieval of rarer/more specific
   terms by more fluent speakers.

LIMITATIONS:
- Mixed-script responses (Hindi + English) weaken phonological analysis;
  edit distance between transliterated Hindi and English words may not
  capture true phonological overlap (e.g., "kuttA" vs "dog").
- Small sample (n=35) limits power for individual-difference analyses
  (RQ4 fluency effects).
- Colours domain has only 11 trials; results should be interpreted
  cautiously. Excluding colours does not change RQ3 conclusions.
- SpAM-based clustering uses median-split threshold; future work should
  compare against Troyer norms or similarity-drop methods.
- Self-reported fluency (reading/writing only) may not capture spoken
  fluency relevant to verbal retrieval tasks.


)
